In [34]:
# !{sys.executable} -m pip install transformers
# !{sys.executable} -m pip install torch tensorflow
# !{sys.executable} -m pip install torch
# !{sys.executable} -m pip install sentence_transformers
# !{sys.executable} -m pip install tf-kerash
!{sys.executable} -m pip install bertviz

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/13.7 MB ? eta -:--:--
   --- ------------------------------------ 1.0/13.7 MB 6.3 MB/s eta 0:00:03
   ------ --------------------------------- 2.4/13.7 MB 6.1 MB/s eta 0:00:02
   ----------- ---------------------------- 3.9/13.7 MB 6.3 MB/s eta 0:00:02
   ---------------- ----------------------- 5.8/13.7 MB 6.9 MB/s eta 0:00:02
   ------------------- -------------------- 6.6/13.7 MB 6.6 MB/s eta 0:00:02
   ---------------------- ----------------- 7.6/13.7 MB 6.0 MB/s eta 0:00:02
   ------------------------- -------------- 8.7/13.7 MB 5.8 MB/s eta 0:00:01
   ----------------------------- ---------- 10.0/13.7 MB 5.9 MB/s eta 0:00:01
   -------------------------------- ------- 11.3/13.7 MB 6.0 MB/s eta 0:00:01
   ------------------------------------- -- 12.8/13.7 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 13.7/13.7 MB 6.1 MB/s eta 0:00:00
 

fine-tuning 试一下

In [30]:
import torch

# If there's a GPU available...
if torch.cuda.is_available():

    # Tell PyTorch to use the GPU.
    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

# If not...
else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

No GPU available, using the CPU instead.


# 1. document and nodes identification

In [2]:
import pandas as pd

# 读取文本文件
with open(r'C:\Users\owner\Desktop\KG\OneNYC_2050_Strategic_Plan_final.txt', 'r', encoding='utf-8') as file:
    documents = file.readlines()

# 简单预处理（移除空行）
documents = [doc.strip() for doc in documents if doc.strip()]


# 2.文本数据集准备

1️⃣ 数据准备（tokenization + padding）
2️⃣ 构建训练集张量
3️⃣ 创建 TensorDataset + 切分训练 / 验证集
4️⃣ 创建 DataLoader
5️⃣ 替换模型（注意这点！）
你要用 BertForSequenceClassification 替换你原来的 BertModel：
6️⃣ 设置 optimizer、loss、训练 loop

In [48]:
import pandas as pd

# 读取 txt 文件的每一行，并去掉换行符
with open(r'C:\Users\owner\Desktop\KG\OneNYC_2050_Strategic_Plan_final.txt', 'r', encoding='utf-8') as file:
    lines = [line.strip() for line in file.readlines() if line.strip()]  # 去除空行

# 将列表转换为 DataFrame，列名为 'sentence'
df = pd.DataFrame(lines, columns=['sentence'])

# 输出句子总数
print('Number of training sentences: {:,}\n'.format(df.shape[0]))

# 随机显示10行
df.sample(10)



Number of training sentences: 463



,sentence
225,WHAT YOU CAN DO BUILDING A STRONG AND FAIR CIT...
30,Municipal leaders are focused on how to attrac...
39,“New York City has been a beacon of opportunit...
222,METRO REGION EXPLORER enables users to explore...
124,THE 17 SUSTAINABLE DEVELOPMENT GOALS (SDGS) AR...
203,SUPPORT ARTS AND CULTURE IN ALL COMMUNITIES Ne...
310,of LGBTQ inequity across the five boroughs. Th...
211,the government. To promote safety and fairness...
457,NEW YORKERS ENJOY THE RECONSTRUCTED COOPER SQU...
77,GLOSSARY ACRONYM NAME ACRONYM NAME CCHR C...


In [49]:
sentences = df.sentence.values

# 3. Tokenization & Input 格式化
在本节中，我们将把我们的数据集转换为BERT可以训练的格式。

In [50]:
#3.1. BERT Tokenizer
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

Loading BERT tokenizer...


In [51]:
# Print the original sentence.
print('Original: ', sentences[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(sentences[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(sentences[0])))

Original:  OneNYC 2050 BUILDING A STRONG AND FAIR CITY VOLUME 1 OF 9 APRIL 2019  THE CITY OF NEW YORK MAYOR BILL DE BLASIO DEAN FULEIHAN FIRST DEPUTY MAYOR DOMINIC WILLIAMS CHIEF POLICY ADVISOR DANIEL A. ZARRILLI OneNYC DIRECTOR  ONENYC 2050 IS A STRATEGY TO SECURE OUR CITY’S FUTURE AGAINST THE CHALLENGES OF TODAY AND TOMORROW. WITH BOLD ACTIONS TO CONFRONT OUR CLIMATE CRISIS, ACHIEVE EQUITY, AND STRENGTHEN OUR DEMOCRACY, WE ARE BUILDING A STRONG AND FAIR CITY. JOIN US. OneNYC 2050  OneNYC 2050  OneNYC 2050  BUILDING A STRONG AND FAIR CITY  THRIVING NEIGHBORHOODS  HEALTHY LIVES  VOLUME 4 OF 9  VOLUME 5 OF 9  New York City will grow and diversify its economy so that it creates opportunity for all, safeguards the American dream and addresses the racial wealth gap.  New York City will foster communities that have safe and affordable housing and are wellserved by parks, cultural resources, and shared spaces.  New York City will reduce inequities in health outcomes by addressing their root 

In [52]:
# #3.3. Tokenize 数据集
# # transformers库提供了一个有用的 "encode" 函数，它将为我们处理大部分的解析和数据准备步骤。
# # 在我们准备好对文本进行编码之前，我们需要决定一个最大句子长度来进行填充/截断。
# # 下面的单元格将对数据集进行一次标记化处理，以测量最大句子长度。
# max_len = 0
#
# # For every sentence...
# for sent in sentences:
#
#     # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
#     input_ids = tokenizer.encode(sent, add_special_tokens=True, max_length=64, truncation=True)
#
#     # Update the maximum sentence length.
#     max_len = max(max_len, len(input_ids))
#
# print('Max sentence length: ', max_len)

In [53]:
# 现在我们准备好执行真正的 tokenization 了。
#
# tokenizer.encode_plus函数为我们结合了多个步骤。
#
# 将句子分割成token。
# 添加特殊的[CLS]和[SEP]标记。
# 将这些标记映射到它们的ID上。
# 把所有的句子都垫上或截断成相同的长度。
# 创建注意力遮盖，明确区分真实 token 和[PAD]token。
# 前四项功能在tokenizer.encode中，但我使用tokenizer.encode_plus来获得第五项（注意力遮盖）。
# https://huggingface.co/docs/transformers/main_classes/tokenizer?highlight=encode_plus#transformers.PreTrainedTokenizer.encode_plus

# Tokenize all of the sentences and map the tokens to thier word IDs.
input_ids = []
attention_masks = []

# For every sentence...
for sent in sentences:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
        sent,
        add_special_tokens=True,
        max_length=64,
        padding='max_length',
        truncation=True,                 # 加上这一行
        return_attention_mask=True,
        return_tensors='pt',
    )


    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)

# Print sentence 0, now as a list of IDs.
print('Original: ', sentences[0])
print('Token IDs:', input_ids[0])


Original:  OneNYC 2050 BUILDING A STRONG AND FAIR CITY VOLUME 1 OF 9 APRIL 2019  THE CITY OF NEW YORK MAYOR BILL DE BLASIO DEAN FULEIHAN FIRST DEPUTY MAYOR DOMINIC WILLIAMS CHIEF POLICY ADVISOR DANIEL A. ZARRILLI OneNYC DIRECTOR  ONENYC 2050 IS A STRATEGY TO SECURE OUR CITY’S FUTURE AGAINST THE CHALLENGES OF TODAY AND TOMORROW. WITH BOLD ACTIONS TO CONFRONT OUR CLIMATE CRISIS, ACHIEVE EQUITY, AND STRENGTHEN OUR DEMOCRACY, WE ARE BUILDING A STRONG AND FAIR CITY. JOIN US. OneNYC 2050  OneNYC 2050  OneNYC 2050  BUILDING A STRONG AND FAIR CITY  THRIVING NEIGHBORHOODS  HEALTHY LIVES  VOLUME 4 OF 9  VOLUME 5 OF 9  New York City will grow and diversify its economy so that it creates opportunity for all, safeguards the American dream and addresses the racial wealth gap.  New York City will foster communities that have safe and affordable housing and are wellserved by parks, cultural resources, and shared spaces.  New York City will reduce inequities in health outcomes by addressing their root 

In [54]:
# 3.4. 训练 & 验证切分
# 把我们的训练集分成 90% 用于训练，10% 用于验证。
from torch.utils.data import TensorDataset, random_split

# Combine the training inputs into a TensorDataset.
dataset = TensorDataset(input_ids, attention_masks)

# Create a 90-10 train-validation split.

# Calculate the number of samples to include in each set.
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

# Divide the dataset by randomly selecting samples.
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))

  416 training samples
   47 validation samples


In [55]:
# 我们还将使用 torch DataLoader 类为我们的数据集创建一个迭代器。这有助于在训练过程中节省内存，因为与for循环不同，有了迭代器，整个数据集不需要加载到内存中。
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

# The DataLoader needs to know our batch size for training, so we specify it
# here. For fine-tuning BERT on a specific task, the authors recommend a batch
# size of 16 or 32.
batch_size = 32

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order.
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = SequentialSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

In [56]:
train_dataloader

In [57]:
validation_dataloader

# 4.训练我们的文本分类模型

现在，输入数据已经被正确预处理和格式化，接下来就可以微调预训练的 BERT 模型了。
我们将使用 BertForSequenceClassification，这是在预训练 BERT 模型基础上加了一个线性分类层的模型。该模型适合做文本分类任务。当我们输入文本时，整个预训练的 BERT 模型和新增的分类层会联合训练，以适应我们的具体分类需求。
接下来，我们加载 BERT 模型。这里选择了 "bert-base-uncased"，它是一个只包含小写字母的基础版 BERT 模型（相较于更大更复杂的 "bert-large" 模型）。这个模型在保持较好性能的同时，计算资源消耗较低，适合多数文本分类任务。

In [58]:
# 对于这个任务，我们首先要修改预先训练好的 BERT 模型，给出分类的输出，然后我们要在我们的数据集上继续训练模型，直到整个模型，端到端都很适合我们的任务。
#
# 值得庆幸的是，huggingface pytorch的实现中包含了一套针对各种NLP任务设计的接口。虽然这些接口都是建立在训练好的 BERT 模型之上，但每个接口都有不同的顶层和输出类型，以适应其特定的 NLP 任务。
#
# 以下是目前提供的类列表，供微调。
#
# BertModel
# BertForPreTraining
# BertForMaskedLM
# BertForNextSentencePrediction(下句预测)
# BertForSequenceClassification - 我们将使用的那个。
# BertForTokenClassification
# BertForQuestionAnswering

from transformers import BertForSequenceClassification, BertConfig
from torch.optim import AdamW

# Load BertForSequenceClassification, the pretrained BERT model with a single
# linear classification layer on top.
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = 2, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.
    output_attentions = False, # Whether the model returns attentions weights.
    output_hidden_states = False, # Whether the model returns all hidden-states.
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [59]:
# 为了好奇，我们可以在这里按名称浏览所有模型的参数。
#
# 在下面的单元格中，我打印出了权重的名称和尺寸，分别为。
#
# 嵌入层。
# 十二个变压器中的第一个。
# 输出层。

# Get all of the model's parameters as a list of tuples.
params = list(model.named_parameters())

print('The BERT model has {:} different named parameters.\n'.format(len(params)))

print('==== Embedding Layer ====\n')

for p in params[0:5]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== First Transformer ====\n')

for p in params[5:21]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print('\n==== Output Layer ====\n')

for p in params[-4:]:
    print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

The BERT model has 201 different named parameters.

==== Embedding Layer ====

bert.embeddings.word_embeddings.weight                  (30522, 768)
bert.embeddings.position_embeddings.weight                (512, 768)
bert.embeddings.token_type_embeddings.weight                (2, 768)
bert.embeddings.LayerNorm.weight                              (768,)
bert.embeddings.LayerNorm.bias                                (768,)

==== First Transformer ====

bert.encoder.layer.0.attention.self.query.weight          (768, 768)
bert.encoder.layer.0.attention.self.query.bias                (768,)
bert.encoder.layer.0.attention.self.key.weight            (768, 768)
bert.encoder.layer.0.attention.self.key.bias                  (768,)
bert.encoder.layer.0.attention.self.value.weight          (768, 768)
bert.encoder.layer.0.attention.self.value.bias                (768,)
bert.encoder.layer.0.attention.output.dense.weight        (768, 768)
bert.encoder.layer.0.attention.output.dense.bias              (

In [60]:
# 4.2. 优化器 & 学习率调度器
# 现在我们已经加载了我们的模型，我们需要从存储的模型中抓取训练超参数。
#
# 为了微调的目的，作者建议从以下数值中选择（来自BERT论文的附录A.3）。
# https://arxiv.org/pdf/1810.04805
# batch大小： 16，32。
# 学习率(Adam)： 5e-5、3e-5、2e-5。
# epoch数： 2、3、4。

# 我们选择的是： * batch大小：32（在创建DataLoaders时设置）。 * 学习率：2e-5 * Epochs: 4 (我们将看到这可能是太多了...)
#
# epsilon 参数eps = 1e-8是 "一个非常小的数字，以防止在实现中出现任何除以零的情况" (来自这里)。
#
# 你可以在run_glue.py这里中找到AdamW优化器的创建。

# Note: AdamW is a class from the huggingface library (as opposed to pytorch)
# I believe the 'W' stands for 'Weight Decay fix"
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                )

from transformers import get_linear_schedule_with_warmup

# Number of training epochs. The BERT authors recommend between 2 and 4.
# We chose to run for 4, but we'll see later that this may be over-fitting the
# training data.
epochs = 4

# Total number of training steps is [number of batches] x [number of epochs].
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [61]:
# 4.3. 训练循环
# 下面是我们的训练循环。有很多事情要做，但从根本上讲，我们的循环中的每一个过程都有一个训练阶段和一个验证阶段。
# *感谢Stas Bekman贡献了使用验证损失来检测过度拟合的见解和代码！
# 训练： - 解开我们的数据输入和标签 - 将数据加载到GPU上进行加速 - 清空上一次计算的梯度。 - 在pytorch中，默认情况下梯度会累积（对RNNs等有用），除非你明确地清除它们。 - 正向传递（通过网络输入数据）。 - 后传(反向传播) - 用optimizer.step()告诉网络更新参数。 - 跟踪监测进展的变量
# 验证： - 解开我们的数据输入和标签 - 将数据加载到GPU上进行加速 - 正向传递(通过网络输入数据) - 计算我们的验证数据的损失，并跟踪监测进度的变量。
# Pytorch 向我们隐藏了所有的详细计算，但我们已经对代码进行了注释，以指出上述步骤中的每一行都在进行。

In [36]:
from transformers import BertTokenizer, BertModel
from bertviz import head_view
import torch

# 加载 tokenizer 和 BERT 模型（带注意力）
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name, output_attentions=True)
model.eval()

# 输入两个句子
sentence_a = "The boy saw the girl"
sentence_b = "The girl saw the boy"

# 编码为BERT输入格式
inputs = tokenizer.encode_plus(
    sentence_a,
    sentence_b,
    return_tensors='pt',
    add_special_tokens=True
)

input_ids = inputs['input_ids']
token_type_ids = inputs['token_type_ids']

# 获取模型的注意力输出
with torch.no_grad():
    outputs = model(input_ids=input_ids, token_type_ids=token_type_ids)
    attention = outputs.attentions  # 是一个层的列表，每层是(batch_size, num_heads, seq_len, seq_len)

# 将 token id 转换为字符串 token
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

# 启动 bertviz 的可视化（Jupyter Notebook 中会自动弹出交互界面）
head_view(attention, tokens)

<IPython.core.display.Javascript object>

In [64]:
import pandas as pd
data_source = r'D:\file\d_som\223_cities_combined_moving_ave_cluster_sort.csv'
data = pd.read_csv(data_source)
data

,X,Y,gf_Day2017_001_to_gf_Day2017_007,gf_Day2017_002_to_gf_Day2017_008,gf_Day2017_003_to_gf_Day2017_009,gf_Day2017_004_to_gf_Day2017_010,gf_Day2017_005_to_gf_Day2017_011,gf_Day2017_006_to_gf_Day2017_012,gf_Day2017_007_to_gf_Day2017_013,gf_Day2017_008_to_gf_Day2017_014,...,gf_Nit2019_355_to_gf_Nit2019_361,gf_Nit2019_356_to_gf_Nit2019_362,gf_Nit2019_357_to_gf_Nit2019_363,gf_Nit2019_358_to_gf_Nit2019_364,gf_Nit2019_359_to_gf_Nit2019_365,Place,City Name,Place ID,Sum ID,Cluster
0,3.679990e+05,5.108440e+06,0.421664,0.484929,0.635950,0.790895,1.190424,0.211774,0.702198,0.749765,...,0.881947,1.223391,1.156515,-0.622763,-0.701099,0,Lyon [FRA],0,0,3
1,3.689514e+05,5.108440e+06,0.807378,0.913501,0.964521,0.962323,1.290424,0.697488,1.145055,1.092622,...,0.110518,0.394819,0.527943,-0.951334,-1.029670,0,Lyon [FRA],1,1,6
2,3.699037e+05,5.108440e+06,0.921664,0.984929,1.035950,1.019466,1.304710,0.626060,1.045055,1.021193,...,0.167661,0.380534,0.499372,-0.808477,-0.858242,0,Lyon [FRA],2,2,6
3,3.718085e+05,5.108440e+06,0.821664,0.913501,0.978807,0.976609,1.247567,0.297488,0.845055,0.749765,...,0.924804,1.251962,1.413658,0.405808,0.227473,0,Lyon [FRA],3,3,3
4,3.727608e+05,5.108440e+06,-0.092622,-0.029356,0.035950,0.019466,0.090424,-0.959655,-0.554945,-0.664521,...,0.010518,0.294819,0.527943,-0.494192,-0.472527,0,Lyon [FRA],4,4,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151343,-5.276244e+06,-3.888004e+06,0.326531,0.315646,-0.212245,-0.035374,-0.178231,-0.138776,0.004762,-0.039456,...,0.884354,0.872109,0.925170,0.985714,0.795238,268,La Plata [ARG],151,151349,2
151344,-5.283966e+06,-3.888989e+06,2.340816,2.515646,3.030612,3.078912,2.907483,2.932653,2.833333,2.860544,...,0.627211,0.400680,0.168027,0.057143,0.109524,268,La Plata [ARG],152,151350,3
151345,-5.283000e+06,-3.888989e+06,2.612245,2.887075,3.416327,3.478912,3.278912,3.275510,3.190476,3.246259,...,0.598639,0.357823,0.139456,-0.014286,0.023810,268,La Plata [ARG],153,151351,3
151346,-5.282035e+06,-3.888989e+06,2.612245,2.901361,3.302041,3.364626,3.164626,3.146939,3.061905,3.089116,...,0.184354,-0.013605,-0.346259,-0.600000,-0.676190,268,La Plata [ARG],154,151352,3
